# PTU Sizing Demo Notebook

This notebook packages the indicative PTU workshop/demo logic into a reusable standalone artifact. It estimates baseline PTUs, compares PTU vs PAYGO monthly cost assumptions, and returns an architecture suggestion.

**Important:** This is not the official PTU calculator. Replace model throughput, minimum commit, and pricing assumptions with validated values before customer use.


In [ ]:
import math
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

DEFAULTS = {
    'avg_rpm': 60,
    'avg_input_tokens': 1800,
    'avg_output_tokens': 650,
    'p95_multiplier': 1.8,
    'cache_rate': 0.20,
    'model_tpm_per_ptu': 3000,
    'output_weight': 4.0,
    'baseline_load_factor': 0.70,
    'safety_buffer': 0.15,
    'min_ptu_commit': 15,
    'ptu_hourly_price': 15.0,
    'paygo_input_per_1m': 5.0,
    'paygo_output_per_1m': 15.0,
    'hours_per_month': 730,
}

DEFAULTS


In [ ]:
def calculate(values):
    avg_tpm = values['avg_rpm'] * (
        values['avg_input_tokens'] * (1 - values['cache_rate']) +
        values['avg_output_tokens'] * values['output_weight']
    )
    p95_tpm = avg_tpm * values['p95_multiplier']
    baseline_tpm = p95_tpm * values['baseline_load_factor']
    raw_baseline_ptu = baseline_tpm / max(values['model_tpm_per_ptu'], 1)
    recommended_ptu = max(
        math.ceil(raw_baseline_ptu * (1 + values['safety_buffer'])),
        max(math.ceil(values['min_ptu_commit']), 0)
    )
    peak_reference_ptu = math.ceil((p95_tpm / max(values['model_tpm_per_ptu'], 1)) * (1 + values['safety_buffer']))
    burst_ratio = (p95_tpm / baseline_tpm) if baseline_tpm > 0 else 0

    monthly_requests = values['avg_rpm'] * 60 * values['hours_per_month']
    input_tokens_monthly = monthly_requests * values['avg_input_tokens'] * (1 - values['cache_rate'])
    output_tokens_monthly = monthly_requests * values['avg_output_tokens']
    paygo_monthly = (input_tokens_monthly / 1_000_000) * values['paygo_input_per_1m'] + (output_tokens_monthly / 1_000_000) * values['paygo_output_per_1m']
    ptu_monthly = recommended_ptu * values['ptu_hourly_price'] * values['hours_per_month']

    architecture = {
        'label': 'PTU + Standard spillover',
        'summary': 'Recommended default for enterprise production: size PTU for the steady-state baseline and keep Standard available for bursts and overflow.',
        'reason': 'Your burst profile suggests a baseline layer plus elasticity is safer than sizing PTU for every short-lived peak.'
    }
    if burst_ratio < 2:
        architecture = {
            'label': 'PTU-first production baseline',
            'summary': 'Your workload looks relatively steady. PTU is likely a good fit for the primary production layer, then validate on hourly PTU before reservation.',
            'reason': 'Lower peak-to-mean burstiness usually aligns better with PTU economics and more predictable throughput.'
        }
    elif burst_ratio >= 4:
        architecture = {
            'label': 'PAYGO or smaller PTU pilot',
            'summary': 'Your burstiness is high enough that PAYGO can remain the better default, or you can test a smaller PTU baseline and let Standard handle most spikes.',
            'reason': 'Very high peak-to-mean ratios often push customers toward PAYGO economics unless the steady baseline is still large.'
        }

    return {
        'avg_tpm': avg_tpm,
        'p95_tpm': p95_tpm,
        'baseline_tpm': baseline_tpm,
        'raw_baseline_ptu': raw_baseline_ptu,
        'recommended_ptu': recommended_ptu,
        'peak_reference_ptu': peak_reference_ptu,
        'burst_ratio': burst_ratio,
        'monthly_requests': monthly_requests,
        'input_tokens_monthly': input_tokens_monthly,
        'output_tokens_monthly': output_tokens_monthly,
        'paygo_monthly': paygo_monthly,
        'ptu_monthly': ptu_monthly,
        'savings_delta': paygo_monthly - ptu_monthly,
        'architecture': architecture,
        'reservation_note': 'Reservation should be treated as a billing optimization after workload validation, not as the first step and not as capacity by itself.'
    }


## Edit inputs and run

Update any values below for a customer scenario, then run the next cells.


In [ ]:
inputs = DEFAULTS.copy()
inputs.update({
    'avg_rpm': 60,
    'avg_input_tokens': 1800,
    'avg_output_tokens': 650,
    'p95_multiplier': 1.8,
})
inputs


In [ ]:
result = calculate(inputs)
summary = pd.DataFrame([
    ('Recommended PTUs', result['recommended_ptu']),
    ('Peak reference PTUs', result['peak_reference_ptu']),
    ('Baseline input-equivalent TPM', result['baseline_tpm']),
    ('P95 input-equivalent TPM', result['p95_tpm']),
    ('PTU monthly', result['ptu_monthly']),
    ('PAYGO monthly', result['paygo_monthly']),
    ('Burst ratio', result['burst_ratio']),
], columns=['Metric', 'Value'])
summary


In [ ]:
display(Markdown(f"### {result['architecture']['label']}"))
display(Markdown(result['architecture']['summary']))
display(Markdown(f"**Why:** {result['architecture']['reason']}"))
display(Markdown(f"**Reservation note:** {result['reservation_note']}"))


In [ ]:
chart_df = pd.DataFrame({
    'Scenario': ['Baseline PTU', 'Peak ref PTU'],
    'PTUs': [result['recommended_ptu'], result['peak_reference_ptu']]
})
ax = chart_df.plot(kind='bar', x='Scenario', y='PTUs', legend=False, figsize=(8, 4), color=['#2563EB', '#94A3B8'])
ax.set_title('Baseline vs peak PTU view')
ax.set_ylabel('PTUs')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## Optional widget UI

If `ipywidgets` is available, run the cell below for a lightweight interactive notebook experience.


In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import clear_output

    avg_rpm_w = widgets.FloatText(value=DEFAULTS['avg_rpm'], description='Avg RPM')
    in_w = widgets.FloatText(value=DEFAULTS['avg_input_tokens'], description='In tokens')
    out_w = widgets.FloatText(value=DEFAULTS['avg_output_tokens'], description='Out tokens')
    p95_w = widgets.FloatSlider(value=DEFAULTS['p95_multiplier'], min=1, max=5, step=0.1, description='P95 x')
    cache_w = widgets.FloatSlider(value=DEFAULTS['cache_rate'], min=0, max=0.9, step=0.05, description='Cache')
    btn = widgets.Button(description='Recalculate', button_style='primary')
    out = widgets.Output()

    def refresh(_=None):
        vals = DEFAULTS.copy()
        vals.update({
            'avg_rpm': avg_rpm_w.value,
            'avg_input_tokens': in_w.value,
            'avg_output_tokens': out_w.value,
            'p95_multiplier': p95_w.value,
            'cache_rate': cache_w.value,
        })
        r = calculate(vals)
        with out:
            clear_output()
            display(pd.DataFrame([
                ('Recommended PTUs', r['recommended_ptu']),
                ('PTU monthly', round(r['ptu_monthly'], 2)),
                ('PAYGO monthly', round(r['paygo_monthly'], 2)),
                ('Architecture', r['architecture']['label']),
            ], columns=['Metric', 'Value']))

    btn.on_click(refresh)
    ui = widgets.VBox([widgets.HBox([avg_rpm_w, in_w, out_w]), widgets.HBox([p95_w, cache_w]), btn, out])
    display(ui)
    refresh()
except Exception as exc:
    print('ipywidgets UI not available in this environment:', exc)
